# Planning Agent Example

This notebook demonstrates the Planning design pattern using LangGraph with ReAct (Reasoning + Acting).

## What is Planning?

Planning is when an agent creates a step-by-step plan before execution, then executes each step sequentially.

## Workflow

```
User Query -> Generate Plan -> Execute Step 1 -> Execute Step 2 -> ... -> Final Output
```

## 1. Setup and Imports

In [ ]:
import os
from typing import TypedDict, Annotated, Sequence, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# Optional: Import aisuite for multi-provider support
try:
    import aisuite as ai
    USE_AISUITE = True
except ImportError:
    USE_AISUITE = False
    print("aisuite not installed. Using langchain_openai directly.")

## 2. Define Tools

Tools are functions the agent can call to interact with external systems.

In [ ]:
from datetime import datetime
import json

# Mock database for demonstration
PRODUCTS_DB = {
    "1001": {"name": "Aviator", "style": "pilot", "frame": "metal", "price": 80, "stock": 12},
    "1002": {"name": "Catseye", "style": "vintage", "frame": "plastic", "price": 60, "stock": 28},
    "1003": {"name": "Moon", "style": "round", "frame": "plastic", "price": 120, "stock": 15},
    "1004": {"name": "Classic", "style": "round", "frame": "metal", "price": 60, "stock": 9},
    "1005": {"name": "Sport", "style": "athletic", "frame": "titanium", "price": 150, "stock": 5},
}

ORDERS_DB = {
    "ORD-001": {"customer": "john@email.com", "items": ["1001", "1004"], "total": 140, "status": "delivered"},
    "ORD-002": {"customer": "jane@email.com", "items": ["1003"], "total": 120, "status": "shipped"},
    "ORD-003": {"customer": "bob@email.com", "items": ["1002", "1005"], "total": 210, "status": "processing"},
}

def get_product_info(product_id: str) -> dict:
    """
    Get product information by product ID.
    
    Args:
        product_id: The product ID (e.g., '1001')
    
    Returns:
        Product information dictionary
    """
    return PRODUCTS_DB.get(product_id, {"error": "Product not found"})

def search_products(style: str = None, max_price: int = None) -> list:
    """
    Search for products matching criteria.
    
    Args:
        style: Product style to filter by (e.g., 'round', 'vintage')
        max_price: Maximum price filter
    
    Returns:
        List of matching products
    """
    results = []
    for pid, product in PRODUCTS_DB.items():
        match = True
        if style and product["style"] != style.lower():
            match = False
        if max_price and product["price"] > max_price:
            match = False
        if match:
            results.append({"id": pid, **product})
    return results

def check_inventory(product_id: str) -> dict:
    """
    Check inventory for a specific product.
    
    Args:
        product_id: The product ID to check
    
    Returns:
        Inventory status
    """
    product = PRODUCTS_DB.get(product_id)
    if product:
        return {"product_id": product_id, "in_stock": product["stock"] > 0, "quantity": product["stock"]}
    return {"error": "Product not found"}

def get_order_status(order_id: str) -> dict:
    """
    Get order status by order ID.
    
    Args:
        order_id: The order ID (e.g., 'ORD-001')
    
    Returns:
        Order information
    """
    return ORDERS_DB.get(order_id, {"error": "Order not found"})

def process_return(order_id: str, reason: str) -> dict:
    """
    Process a return request.
    
    Args:
        order_id: The order ID to return
        reason: Reason for return
    
    Returns:
        Return confirmation
    """
    if order_id in ORDERS_DB:
        return {"status": "return_initiated", "order_id": order_id, "refund_amount": ORDERS_DB[order_id]["total"]}
    return {"error": "Order not found"}

# List of all available tools
tools = [get_product_info, search_products, check_inventory, get_order_status, process_return]

print(f"Defined {len(tools)} tools:")
for tool in tools:
    print(f"  - {tool.__name__}: {tool.__doc__.strip().split(chr(10))[0].strip()}")

## 3. Define Agent State

In [ ]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator

class AgentState(TypedDict):
    """
    State for the planning agent.
    
    Attributes:
        messages: List of messages in the conversation
        plan: Current execution plan (if any)
        current_step: Current step being executed
    """
    messages: Annotated[Sequence[BaseMessage], operator.add]
    plan: list
    current_step: int

## 4. Create the Planning Agent with LangGraph

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# Initialize the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Create the ReAct agent with tools
memory = MemorySaver()
agent = create_react_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    state_modifier="""You are a helpful customer service agent for a sunglasses store.

When handling customer queries:
1. First understand what the customer needs
2. Plan your approach - which tools you'll need
3. Execute the plan step by step
4. Provide a clear, helpful response

Available tools:
- get_product_info: Get details about a specific product
- search_products: Search products by style or price
- check_inventory: Check if a product is in stock
- get_order_status: Check order status
- process_return: Initiate a return

Always be helpful and professional."""
)

print("Planning agent created successfully!")

## 5. Example Queries

### Example 1: Product Search

In [ ]:
# Example query: Search for round sunglasses under $100
query1 = "Do you have any round sunglasses in stock that are under $100?"

print(f"Query: {query1}\n")
print("=" * 60)

# Run the agent
config = {"configurable": {"thread_id": "demo-1"}}
result = agent.invoke(
    {"messages": [HumanMessage(content=query1)]},
    config=config
)

# Print the conversation
for msg in result["messages"]:
    if isinstance(msg, HumanMessage):
        print(f"User: {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.content:
            print(f"Agent: {msg.content}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Tool Call] {tc['name']}({tc['args']})")
    else:
        print(f"[Tool Result] {msg.content[:200]}...")

print("\n" + "=" * 60)
print("Final Response:")
print(result["messages"][-1].content)

### Example 2: Order Status and Return

In [ ]:
# Example query: Check order and process return
query2 = "I want to return my order ORD-002. The sunglasses don't fit me well."

print(f"Query: {query2}\n")
print("=" * 60)

# Run the agent
config = {"configurable": {"thread_id": "demo-2"}}
result = agent.invoke(
    {"messages": [HumanMessage(content=query2)]},
    config=config
)

# Print the conversation
for msg in result["messages"]:
    if isinstance(msg, HumanMessage):
        print(f"User: {msg.content}")
    elif isinstance(msg, AIMessage):
        if msg.content:
            print(f"Agent: {msg.content}")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Tool Call] {tc['name']}({tc['args']})")
    else:
        print(f"[Tool Result] {msg.content[:200]}...")

print("\n" + "=" * 60)
print("Final Response:")
print(result["messages"][-1].content)

### Example 3: Multi-Step Planning

In [ ]:
# Complex query requiring multiple steps
query3 = """I'm looking for sunglasses for my upcoming vacation. 
I prefer metal frames and my budget is around $100. 
Can you show me what's available and check if they're in stock?"""

print(f"Query: {query3}\n")
print("=" * 60)

# Run the agent
config = {"configurable": {"thread_id": "demo-3"}}
result = agent.invoke(
    {"messages": [HumanMessage(content=query3)]},
    config=config
)

print("\n" + "=" * 60)
print("Final Response:")
print(result["messages"][-1].content)

## 6. Understanding the Planning Process

Let's trace through what the agent does step by step.

In [ ]:
def trace_agent_execution(query: str, thread_id: str = "trace"):
    """
    Trace the agent's execution step by step.
    """
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}\n")
    
    step = 0
    config = {"configurable": {"thread_id": thread_id}}
    
    for event in agent.stream(
        {"messages": [HumanMessage(content=query)]},
        config=config,
        stream_mode="values"
    ):
        step += 1
        last_msg = event["messages"][-1]
        
        print(f"\n--- Step {step} ---")
        
        if isinstance(last_msg, HumanMessage):
            print(f"[USER] {last_msg.content}")
            
        elif isinstance(last_msg, AIMessage):
            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"[AGENT DECISION] Call tool: {tc['name']}")
                    print(f"[ARGUMENTS] {tc['args']}")
            else:
                print(f"[AGENT RESPONSE] {last_msg.content}")
                
        else:
            # Tool result
            print(f"[TOOL RESULT] {last_msg.content}")

# Run a traced execution
trace_agent_execution("What's the price of the Aviator sunglasses and are they in stock?")

## 7. Planning with Structured Output

For more control, we can have the agent output a structured plan first.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class PlanStep(BaseModel):
    """A single step in the execution plan."""
    step_number: int = Field(description="Step number in the plan")
    action: str = Field(description="Action to take")
    tool: Optional[str] = Field(description="Tool to use, if any")
    expected_output: str = Field(description="What we expect to get from this step")

class ExecutionPlan(BaseModel):
    """A complete execution plan."""
    goal: str = Field(description="The overall goal")
    steps: List[PlanStep] = Field(description="List of steps to execute")
    estimated_tools: List[str] = Field(description="Tools that will be needed")

# Create a planning LLM
planning_llm = llm.with_structured_output(ExecutionPlan)

def create_plan(query: str) -> ExecutionPlan:
    """
    Create a structured execution plan for a query.
    """
    system_prompt = """You are a planning agent for a customer service system.
    
Available tools:
- get_product_info: Get details about a specific product
- search_products: Search products by style or price
- check_inventory: Check if a product is in stock
- get_order_status: Check order status
- process_return: Initiate a return

Create a step-by-step plan to answer the customer's query."""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Create a plan for: {query}")
    ]
    
    return planning_llm.invoke(messages)

# Test the planning
query = "I want to buy round sunglasses under $100. Show me what's available."
plan = create_plan(query)

print(f"Goal: {plan.goal}\n")
print("Steps:")
for step in plan.steps:
    print(f"  {step.step_number}. {step.action}")
    if step.tool:
        print(f"     Tool: {step.tool}")
    print(f"     Expected: {step.expected_output}\n")

## 8. Key Takeaways

### When to Use Planning

| Scenario | Use Planning? |
|----------|---------------|
| Simple lookup | No - direct tool call |
| Multi-step task | Yes |
| Steps depend on previous results | Yes |
| Customer service queries | Yes |
| Complex reasoning required | Yes |

### Planning vs Direct Execution

**Direct Execution (Zero-shot):**
```
Query -> LLM -> Response
```

**Planning:**
```
Query -> Plan -> Execute Steps -> Response
```

### Benefits of Planning

1. **Better handling of complex tasks** - Break down into manageable steps
2. **More transparent** - Can see what the agent is doing
3. **Easier debugging** - Identify which step failed
4. **Better error recovery** - Can retry individual steps

## 9. Exercises

1. **Add a new tool**: Create a tool that recommends sunglasses based on face shape.

2. **Add memory**: Modify the agent to remember previous conversations.

3. **Add human-in-the-loop**: Require human approval before processing returns.

4. **Create a custom planner**: Build a planner that optimizes for minimum tool calls.